<a href="https://colab.research.google.com/github/plrushe/Honours-Code/blob/main/BILSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q uninstall -y numpy pandas scipy scikit-learn tensorflow tf-keras tensorflow-text tensorflow-decision-forests
!pip -q install --upgrade "numpy==1.26.4" "pandas==2.2.2" "scipy==1.11.4" "scikit-learn==1.4.2" "tensorflow==2.19.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-hub 0.21.1 requires tensorflow-text; platform_system != "Windows", which is not installed.
tensorflow-hub 0.16.1 requires tf-keras>=2.14.1, which is not installed.
dopamine-rl 4.1.2 requires tf-keras>=2.18.0, which is not installed.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires scipy>=1.13, but you have scipy 1.11.4 which is incompatible.
libpysal 4.14.1 requires scipy>=1.12.0, but you have scipy 1.11.4 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.11.4 which is incompatible.
esda 2.8.1 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
mapclassify 2.10.0 requires scipy>=1.12, but you have scipy 1.11.4 which is

In [ ]:
import os, sys
os._exit(0)

In [ ]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scipy", __import__("scipy").__version__)
print("sklearn", __import__("sklearn").__version__)
print("tensorflow", tf.__version__)

numpy 1.26.4
pandas 2.2.2
scipy 1.11.4
sklearn 1.4.2
tensorflow 2.19.0


In [ ]:
dataset = pd.read_csv("patient_diaries.csv")

dataset = dataset[["text", "label"]].copy()

norm = dataset["label"].astype(str).str.strip().str.lower()
mapping = {"no depression": 0, "depression": 1}
dataset["label"] = norm.map(mapping)

dataset = dataset.dropna(subset=["text", "label"])
dataset["label"] = dataset["label"].astype("int32")
dataset["text"] = dataset["text"].astype(str)

print("Dataset shape:", dataset.shape)
print(dataset["label"].value_counts())
dataset.head()

Dataset shape: (1000, 2)
label
0    514
1    486
Name: count, dtype: int64


,text,label
0,I felt happy today. I shared a laugh with a fr...,0
1,I felt fine today. I wrote in my journal and e...,1
2,I felt confused today. I felt overwhelmed with...,1
3,I felt confused today. I felt a glimmer of hop...,0
4,I felt down today. I felt a glimmer of hope to...,0


In [ ]:
def basic_clean(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)     # remove URLs
    text = re.sub(r"\s+", " ", text).strip()
    return text

texts = dataset["text"].apply(basic_clean).tolist()
labels = dataset["label"].values

In [ ]:
VOCAB_SIZE = 30000
MAX_LEN = 256
OOV_TOKEN = "<OOV>"

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(texts)

seqs = tokenizer.texts_to_sequences(texts)
X_all = pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")
y_all = labels

print("X_all shape:", X_all.shape, "y_all shape:", y_all.shape)

X_all shape: (1000, 256) y_all shape: (1000,)


In [ ]:
def build_bilstm_model(vocab_size: int, max_len: int) -> tf.keras.Model:
    model = models.Sequential([
        layers.Input(shape=(max_len,)),
        layers.Embedding(input_dim=vocab_size, output_dim=128),
        layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def compute_weights(y):
    classes = np.array([0, 1])
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {0: float(weights[0]), 1: float(weights[1])}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs, precs, recs, f1s = [], [], [], []

EPOCHS = 6
BATCH_SIZE = 32

for fold, (train_idx, val_idx) in enumerate(cv.split(X_all, y_all), start=1):
    X_train, X_val = X_all[train_idx], X_all[val_idx]
    y_train, y_val = y_all[train_idx], y_all[val_idx]

    class_weight = compute_weights(y_train)

    model = build_bilstm_model(VOCAB_SIZE, MAX_LEN)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=[early_stop],
        verbose=0
    )

    probs = model.predict(X_val, batch_size=BATCH_SIZE, verbose=0).ravel()
    preds = (probs >= 0.5).astype(int)

    accs.append(accuracy_score(y_val, preds))
    precs.append(precision_score(y_val, preds, zero_division=0))
    recs.append(recall_score(y_val, preds, zero_division=0))
    f1s.append(f1_score(y_val, preds, zero_division=0))

    print(f"Fold {fold}: acc={accs[-1]:.4f}, prec={precs[-1]:.4f}, rec={recs[-1]:.4f}, f1={f1s[-1]:.4f}")

print("\nCV Summary (mean +/- std)")
print(f"accuracy:  {np.mean(accs):.4f} (+/- {np.std(accs):.4f})")
print(f"precision: {np.mean(precs):.4f} (+/- {np.std(precs):.4f})")
print(f"recall:    {np.mean(recs):.4f} (+/- {np.std(recs):.4f})")
print(f"f1:        {np.mean(f1s):.4f} (+/- {np.std(f1s):.4f})")

Fold 1: acc=0.8950, prec=0.8276, rec=0.9897, f1=0.9014
Fold 2: acc=0.9450, prec=0.9135, rec=0.9794, f1=0.9453


Fold 3: acc=0.9200, prec=0.8716, rec=0.9794, f1=0.9223


Fold 4: acc=0.9300, prec=0.9462, rec=0.9072, f1=0.9263
Fold 5: acc=0.9200, prec=1.0000, rec=0.8367, f1=0.9111

CV Summary (mean +/- std)
accuracy:  0.9220 (+/- 0.0163)
precision: 0.9118 (+/- 0.0595)
recall:    0.9385 (+/- 0.0588)
f1:        0.9213 (+/- 0.0148)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=42
)

class_weight = compute_weights(y_train)

final_model = build_bilstm_model(VOCAB_SIZE, MAX_LEN)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)

final_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

probs = final_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
preds = (probs >= 0.5).astype(int)

print("\nHold-out test results")
print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds, zero_division=0))
print("Recall:", recall_score(y_test, preds, zero_division=0))
print("F1:", f1_score(y_test, preds, zero_division=0))

print("\nClassification report:\n", classification_report(y_test, preds, digits=4))
print("\nConfusion matrix:\n", confusion_matrix(y_test, preds))

Epoch 1/6
20/20 ━━━━━━━━━━━━━━━━━━━━ 14s 418ms/step - accuracy: 0.5495 - loss: 0.6873 - val_accuracy: 0.7563 - val_loss: 0.6233
Epoch 2/6
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 328ms/step - accuracy: 0.7787 - loss: 0.5449 - val_accuracy: 0.8188 - val_loss: 0.4038
Epoch 3/6
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 399ms/step - accuracy: 0.8060 - loss: 0.3729 - val_accuracy: 0.7812 - val_loss: 0.3693
Epoch 4/6
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 386ms/step - accuracy: 0.8528 - loss: 0.3189 - val_accuracy: 0.8938 - val_loss: 0.2635
Epoch 5/6
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 333ms/step - accuracy: 0.9252 - loss: 0.1793 - val_accuracy: 0.9125 - val_loss: 0.1833
Epoch 6/6
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 335ms/step - accuracy: 0.9203 - loss: 0.1639 - val_accuracy: 0.9000 - val_loss: 0.1680

Hold-out test results
Accuracy: 0.91
Precision: 0.8910891089108911
Recall: 0.9278350515463918
F1: 0.9090909090909091

Classification report:
               precision    recall  f1-score   support

           0     0.9293    0.8932    

In [ ]:
import json
os.makedirs("dl_models", exist_ok=True)

final_model.save("dl_models/bilstm_model.keras")

token_json = tokenizer.to_json()
with open("dl_models/tokenizer.json", "w") as f:
    f.write(token_json)

meta = {
    "vocab_size": VOCAB_SIZE,
    "max_len": MAX_LEN,
    "threshold": 0.5,
    "label_mapping": {"no depression": 0, "depression": 1},
}
with open("dl_models/meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved deep learning model + tokenizer + meta in ./dl_models/")

Saved deep learning model + tokenizer + meta in ./dl_models/
